A. Predict the price of the Uber ride from a given pickup point to the agreed drop-off
location. Perform following tasks:
1. Pre-process the dataset.
2. Identify outliers.
3. Check the correlation.
4. Implement linear regression and ridge, Lasso regression models.
5. Evaluate the models and compare their respective scores like R2, RMSE, 

In [1]:
#run once, used to calculate distance between two coordinates
!pip install geopy
!pip install pyproj

  Obtaining dependency information for geopy from https://files.pythonhosted.org/packages/66/e5/de8b557b279e31cdfb2070ac83eb45dc7e29bd93cd544128756f658e52c7/geopy-2.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for geographiclib<3,>=1.52 from https://files.pythonhosted.org/packages/31/b3/802576f2ea5dcb48501bb162e4c7b7b3ca5654a42b2c968ef98a797a4c79/geographiclib-2.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/114.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 kB ? eta -:--:--
   ---------- ----------------------------- 30.7/114.6 kB 1.4 MB/s eta 0:00:01
   ---------- ----------------------------- 30.7/114.6 kB 1.4 MB/s eta 0:00:01
   ------------- ------------------------- 41.0/114.6 kB 281.8 kB/s eta 0:00:01
   ------------- ------------------------- 41.0/114.6 kB 281.8 kB/s eta 0:00:01
   -------------------- ------------------ 61.4/114.6 kB

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error


import time
from datetime import datetime

# Load dataset
df = pd.read_csv("./uber.csv")
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         200000 non-null  int64  
 1   key                200000 non-null  object 
 2   fare_amount        200000 non-null  float64
 3   pickup_datetime    200000 non-null  object 
 4   pickup_longitude   200000 non-null  float64
 5   pickup_latitude    200000 non-null  float64
 6   dropoff_longitude  199999 non-null  float64
 7   dropoff_latitude   199999 non-null  float64
 8   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 13.7+ MB


In [5]:
df = df.drop(columns = ['Unnamed: 0', 'key'])
df = df[df["fare_amount"] > 0]

df["pickup_datetime"] = pd.to_datetime(df['pickup_datetime'])

In [6]:
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["hour"] = df["pickup_datetime"].dt.hour

In [7]:
coordinate_columns = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "fare_amount"
]

for col in coordinate_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[
        (df[col] >= lower) &
        (df[col] <= upper)
    ]

In [8]:
from pyproj import Geod
geod = Geod(ellps="WGS84")

lon1, lat1, lon2, lat2 = df["pickup_longitude"].values, df["pickup_latitude"].values, df["dropoff_longitude"].values, df["dropoff_latitude"].values
temp1, temp2, distance = geod.inv(lon1, lat1, lon2, lat2)
df["distance"] = distance / 1000  # Convert to kilometers
df.drop(columns = ['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude'], inplace = True)
df.dropna(inplace=True)
df.head()


,fare_amount,passenger_count,day,month,year,hour,distance
0,7.5,1,7,5,2015,19,1.681111
1,7.7,1,17,7,2009,20,2.454363
2,12.9,1,24,8,2009,21,5.039603
3,5.3,3,26,6,2009,8,1.661442
5,4.9,1,12,2,2011,2,0.000000


In [9]:
print("Correlation Matrix:")
print(df.corr(numeric_only=True))


Correlation Matrix:
                 fare_amount  passenger_count       day     month      year  \
fare_amount         1.000000         0.017847  0.006096  0.027565  0.138518   
passenger_count     0.017847         1.000000  0.002748  0.009827  0.002504   
day                 0.006096         0.002748  1.000000 -0.016723 -0.011795   
month               0.027565         0.009827 -0.016723  1.000000 -0.116050   
year                0.138518         0.002504 -0.011795 -0.116050  1.000000   
hour                0.010813         0.013553  0.003661 -0.003110  0.004557   
distance            0.794710         0.004781  0.007336 -0.000170 -0.040723   

                     hour  distance  
fare_amount      0.010813  0.794710  
passenger_count  0.013553  0.004781  
day              0.003661  0.007336  
month           -0.003110 -0.000170  
year             0.004557 -0.040723  
hour             1.000000 -0.014284  
distance        -0.014284  1.000000  


In [10]:
X = df[
    [
        "distance",
        "passenger_count",
        'day',
        'month',
        'year',
        'hour'
    ]
]

y = df["fare_amount"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso()
}

In [15]:
for name, model in models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print("\n", name)
    print("R2 Score:", r2)
    print("RMSE:", rmse)


 Linear Regression
R2 Score: 0.6645427969147395
RMSE: 2.0065350871093535

 Ridge Regression
R2 Score: 0.6645428518685612
RMSE: 2.0065349227563827

 Lasso Regression
R2 Score: 0.5514642919110018
RMSE: 2.3202068738549864
